# APIs: Basics & JSON
### OIM 7502 — API Series, Part 1 of 4

Most API interactions boil down to getting or posting data, almost always exchanged as JSON. Every API involves connecting to some endpoint, and most require registering for an API key. Many platforms also publish convenience libraries so you don't have to hand-build requests yourself — Facebook, Spotify, and Yahoo Finance (`yfinance`) all have one, and we'll use `yfinance` later in this series.

In [ ]:
import json
import numpy as np
import pandas as pd
import requests

### Making a request
`dummyjson.com` is a free test API — no key required, good for practicing the mechanics without any setup friction.

In [ ]:
endpoint = "https://dummyjson.com/posts"
req = requests.get(endpoint).text
req[:300]  # raw response is just a big JSON string -- text, not yet a usable Python object

### Parsing JSON into Python
`json.loads` turns that JSON string into native Python dicts/lists you can actually work with.

In [ ]:
data = json.loads(req)
type(data)

In [ ]:
data.keys()

### A pagination gotcha
This is worth slowing down on. `len(data['posts'])` tells you how many posts came back **in this response** — not necessarily how many exist.

In [ ]:
len(data['posts'])

In [ ]:
data['total'], data['skip'], data['limit']

**The API returns 251 total posts, but only sends 30 at a time** (`limit`), starting from `skip=0`. If you only look at `len(data['posts'])` and assume that's everything, you'll silently work with 12% of the data and never know it. Any API with a `total`/`limit`/`skip` (or similarly named) field is telling you it paginates — always check for that before assuming a first response is complete.

### Exploring the data (from this first, limited response)

In [ ]:
data['posts'][0]['title']

In [ ]:
for post in data['posts']:
    print(post['title'])

That's only 30 of the 251 titles that actually exist — everything above came from the paginated, limited response.

### Getting everything anyway: a second call
Most paginated APIs give you a way to opt out of pagination — check the docs rather than assuming there isn't one. For `dummyjson`, the docs state it directly: **`limit=0` returns every item in a single response.** That means a second, separate request, with a different endpoint URL.

In [ ]:
endpoint_all = "https://dummyjson.com/posts?limit=0"
req_all = requests.get(endpoint_all).text
data_all = json.loads(req_all)

len(data_all['posts'])  # now matches data['total'] from the first call, all 251

In [ ]:
for post in data_all['posts']:
    print(post['title'])

**Not every API offers a `limit=0` escape hatch** — some cap the max page size and genuinely require you to loop over `skip` values, accumulating results, until you've seen everything. Always check: does the API have an explicit "give me everything" option, or do you need to paginate manually? Assuming either one without checking is how you either overload a server that expected you to page through results, or silently work with partial data.